[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/10_beyond_one_forward.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# Beyond one forward

**Rung 4 to 5 of 5 (after this: the workflow shelf in [`examples/50min/`](../../examples/50min/README.md)).
You are here: a capture as a part of your experiment, not the whole of it.**
Trigger: your field's experiment.

So far a trace was the end of the line: capture, read, compare. Here it is a step inside
something larger. Activations captured over a dataset train a probe; a capture stays connected to
autograd so a loss can flow through it; a replacement module is spliced into the forward pass;
and then the path forks by field, with one cell each for neuroAI, mechanistic interpretability
and model debugging, and a pointer to the full workflow.

> **You'll want to know:** [gradients](06_gradients.ipynb) (`.grad` beside `.out`),
> [extraction over a dataset](04_keep_what_you_need.ipynb) (`tl.extract_dataset`),
> [interventions](05_change_and_rerun.ipynb) (`intervene=tl.when(where, what)`).

In [2]:
import tempfile
import warnings
from pathlib import Path

import torch
from IPython.display import display
from torch import nn

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [3]:
from learn_models import (
    tiny_attention,
    tiny_dataset,
    tiny_mlp,
)  # tiny seeded CPU models; nothing downloads

model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch
data = tiny_dataset(64)  # 64 MLP inputs as one (64, 8) tensor

A linear probe trained on captured activations: extraction over the dataset, then ordinary
PyTorch.

In [4]:
acts = tl.extract_dataset(model, data, ["relu_1_2", "linear_2_3"], batch_size=16, progress=False)
hidden = acts["relu_1_2"]  # (64, 16)
probe = nn.Linear(16, 1)
optimizer = torch.optim.SGD(probe.parameters(), lr=0.05)
losses = []
for step in range(50):
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(probe(hidden).squeeze(-1), data.sum(dim=1))
    loss.backward()
    optimizer.step()
    losses.append(round(loss.item(), 3))
print("probe loss, step 1 ->  step 50:", losses[0], "->", losses[-1])

probe loss, step 1 ->  step 50: 9.705 -> 4.522


## 1. Train through a capture

Saved activations are still connected to autograd, so a loss on any of them reaches the model's
parameters. `backward_ready=True` declares that intent, and the capture keeps what a backward pass
needs. This is how an auxiliary loss on a hidden layer, or a frozen backbone with a trainable
head, is written with a trace in the loop.

In [5]:
model.zero_grad(set_to_none=True)
trace = tl.trace(model, x, capture=tl.options.CaptureOptions(backward_ready=True))
hidden = trace["relu_1_2"].out
print("still in the graph:", hidden.requires_grad)
(hidden.square().mean()).backward()
print(
    "first layer weight grad:",
    tuple(model.in_proj.weight.grad.shape),
    "| norm",
    round(float(model.in_proj.weight.grad.norm()), 4),
)
model.zero_grad(set_to_none=True)

still in the graph: True
first layer weight grad: (16, 8) | norm 0.3407


## 2. Splice a module in

`tl.splice_module(replacement)` is an edit like `tl.zero_ablate()`: at the selected site the
replacement module runs on the site's output and its result takes the output's place. A trained
sparse autoencoder spliced at the ReLU should leave the model's output almost unchanged; this
untrained stand-in does not, and the size of that gap is the test.

In [6]:
stand_in = nn.Sequential(
    nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 16)
)  # the shape of an SAE, untrained
clean = tl.trace(model, x)
spliced = tl.trace(
    model, x, intervene=tl.when(tl.func("relu"), tl.splice_module(stand_in, input_route="out"))
)
print(
    "output moved by:",
    round(float((spliced.reconstruct_output() - clean.reconstruct_output()).detach().norm()), 3),
)
print("clean  :", clean.reconstruct_output()[0].detach())
print("spliced:", spliced.reconstruct_output()[0].detach())

output moved by: 0.97
clean  : tensor([-0.3464,  0.0938,  0.0193, -0.5142])
spliced: tensor([ 0.0579,  0.0618, -0.1056, -0.2215])


## 3. Fork by field

Three one-cell starts. Each runs on the tiny model above; the linked workflow is the full version.

**NeuroAI.** A representational dissimilarity matrix from the activations you already extracted,
and a comparison between two layers' RDMs. Workflows:
[cog_neuro_rdm](../../examples/5min/cog_neuro_rdm.ipynb),
[cog_neuro_extraction](../../examples/50min/cog_neuro_extraction.ipynb).

In [7]:
rdm_hidden = tl.repgeom.rdm(acts["relu_1_2"][:16])  # 16 items, pairwise distances
rdm_output = tl.repgeom.rdm(acts["linear_2_3"][:16])
print(
    "RDM:",
    tuple(rdm_hidden.shape),
    "| hidden vs output RDM correlation:",
    round(float(tl.repgeom.compare_rdms(rdm_hidden, rdm_output)), 3),
)

RDM: (16, 16) | hidden vs output RDM correlation: 0.457


**Mechanistic interpretability.** Two edits you have not met: `tl.steer(direction, strength)`
adds a vector at a site, and `tl.patch_from(other)` writes another run's value into this run at
the same site. Workflows: [steering](../../examples/50min/steering.ipynb),
[causal_trace_recipe](../../examples/50min/causal_trace_recipe.ipynb),
[paired_prompt_patching](../../examples/50min/paired_prompt_patching.ipynb); attribution needs
transcoders and lives in [attribution.md](../../docs/reference/attribution.md).

In [8]:
attention, ids = tiny_attention()
plain = tl.trace(attention, ids)
direction = plain["out_proj"].out[0, -1].detach()  # steer toward the last position's own output
steered = tl.trace(
    attention,
    ids,
    intervene=tl.when(tl.module("out_proj"), tl.steer(direction, strength=4.0, dim=-1)),
)
print(
    "steering moved the logits by:",
    round(float((steered.reconstruct_output() - plain.reconstruct_output()).detach().norm()), 3),
)

other = tl.trace(model, torch.randn(4, 8, generator=torch.Generator().manual_seed(1)))
patched = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.patch_from(other)))
print(
    "patched site equals the donor's:", torch.equal(patched["relu_1_2"].out, other["relu_1_2"].out)
)

steering moved the logits by: 3.941
patched site equals the donor's: True


**Model debugging.** `find_nan` names the first non-finite op; `tl.report.doctor` checks the
environment and the model; `tl.validate` replays the capture against the model. Card:
[find_first_nan](../../examples/5min/find_first_nan.ipynb).

In [9]:
print("non-finite found:", clean.find_nan().found)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # the report imports a progress bar that warns in a bare kernel
    report = tl.report.doctor(model, x)
print("capture faithful:", tl.validate(model, x, scope="forward"))
report  # findings first: rows whose condition was detected or that are not ok, then the clean count

non-finite found: False
capture faithful: True


TorchLens doctor report: learn_models.TinyMLP
PyTorch 2.13.0+cpu; 42 rows: 8 finding(s), 34 clean
- [error] torch backend scorecard (known_broken): stable; 171/175 cells passed, 0 unsupported, 4 failed; pinned 2.13.0, installed 2.13.0+cpu, supported [2.1,...
    fix: first failed cell: c1_reference_parity (random_forward): captured output != direct reference output
- [warning] Runtime capabilities (scope): 77/87 capabilities present; absent: HAS_CHECKPOINT_INTERNAL_HOOK_CLASS, HAS_CODE_POSITIONS, HAS_CODE_QUALNAME, HAS_...
    fix: Inspect capability_snapshot() for unavailable runtime integrations.
- [warning] mlx backend scorecard (known_broken): preview; 161/171 cells passed, 0 unsupported, 10 failed; pinned 0.26.3, installed absent, supported [0.26, ...
    fix: install the framework: pip install "torchlens[mlx]"
- [warning] jax backend scorecard (known_broken): preview; 146/155 cells passed, 0 unsupported, 9 failed; pinned 0.10.2, installed absent, supported [0.4.30,...
    fix: in

## 4. Extend

Three doors, all of them registration rather than subclassing. `tl.facets.register` taught a
vocabulary in 08. `torchlens.appliances.register(name, factory)` hangs a kit of methods off
`trace.using(name)`; TorchLens registers its own kits the same way, and the list below is what
is installed. Custom hooks compose with replay in
[custom_hooks](../../examples/50min/custom_hooks.ipynb). Other array libraries are selected with
`backend=` on the capture call.

In [10]:
tl.appliances.names()

{'mi': 'mechinterp kit: head_scores, qk_circuit, ov_circuit'}

## 5. The five ideas, once more

**Capture** (`tl.trace`, [01](01_capture_and_look.ipynb)), **address** (`trace[...]`,
[02](02_address_and_read.ipynb)), **read** (`.out`, `.summary()`, `.draw()`, `.grad`,
[03](03_see_it.ipynb) and [06](06_gradients.ipynb)), **edit and rerun** (`intervene=`,
[05](05_change_and_rerun.ipynb)), **compare** (`tl.Bundle`, [07](07_compare_runs.ipynb)). The
rest was the same five with more words: what to keep ([04](04_keep_what_you_need.ipynb)), the
architecture's vocabulary ([08](08_semantic_shortcuts.ipynb)), thousands of batches
([09](09_scale.ipynb)), and this notebook.

**One thing about networks:** a probe trained on captured activations is a claim about what a
layer makes linearly available, which is a different question from what the network uses. The
splice in section 2 is how you test the second: replace the representation and see whether the
output survives.

**Go deeper:** the workflow shelf is [`examples/50min/`](../../examples/50min/README.md); the
reference index is [`docs/reference/`](../../docs/reference/); the vocabulary is
[the glossary](../../docs/reference/glossary.md).